# ORCA-X — Google Colab GPU ML Runner

**Use this notebook as the only entry point for heavy ORCA-X ML training/refinements.**

This notebook is deliberately restart-safe: it always moves to `/content` before touching the repository, so deleting `/content/HackHeritage` can never leave the Python process inside a deleted working directory.

### Required runtime
Runtime → Change runtime type → GPU. T4 or L4 are suitable.

In [ ]:
# Cell 1 — configure the Colab workspace. Safe to run repeatedly.
import os, shutil, subprocess, sys
os.chdir('/content')
REPO_URL = 'https://github.com/Sayan260106/HackHeritage.git'
REPO_REF = 'main'
REPO_DIR = '/content/HackHeritage'
os.environ['ORCA_X_DEVICE'] = 'cuda'
os.environ['ORCA_X_N_JOBS'] = '2'
print('Workspace:', os.getcwd())
print('Repository:', REPO_DIR)

In [ ]:
# Cell 2 — clone a clean copy. NEVER delete the repository while cwd is inside it.
os.chdir('/content')
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('Cloned:', os.getcwd())
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# Cell 3 — install only the ML dependencies.
!python -m pip install -q --upgrade pip
!python -m pip install -q -r ml/requirements-colab.txt

In [ ]:
# Cell 4 — verify the actual GPU AND an actual XGBoost CUDA fit.
!nvidia-smi
!python ml/src/colab_preflight.py

## Choose ONE operation below

Do not start several heavy training jobs at once.

In [ ]:
# A — Rebuild the real historical dataset (only needed when the dataset is absent/stale).
# !python ml/src/download_historical_marine.py
# !python ml/src/prepare_dataset.py

In [ ]:
# B — Canonical production model. This uses ORCA_X_DEVICE=cuda.
# !python ml/src/train.py

In [ ]:
# C — Refinement 26. Use the adapter so the existing refinement uses CUDA.
!python ml/src/colab_gpu_runner.py ml/src/refinement26_uncertainty_aware_forecast.py

In [ ]:
# D — Refinement 25. Uncomment only when you want to run it.
# !python ml/src/colab_gpu_runner.py ml/src/refinement25_temporal_reliability_forecast.py

## Other XGBoost scripts

For any existing XGBoost training/refinement script under `ml/src`, run it through the adapter:

`!python ml/src/colab_gpu_runner.py ml/src/<script>.py`

The adapter supports `XGBClassifier`, `XGBRegressor`, and `XGBRanker`. Pandas/NumPy/scikit-learn work remains on Colab CPU; XGBoost tree computation is GPU-accelerated.

In [ ]:
# Optional — inspect generated model/evaluation artifacts.
import os
for root, dirs, files in os.walk('ml/models'):
    for name in files:
        print(os.path.join(root, name))